In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_14_sampling_and_variance_reduction.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Atomic Scale Simulations · Lesson 14: Non-Uniform Sampling, Cholesky Gaussians & Importance Sampling Variance Reduction

_Session 8153735e · Estimated time: ~115 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Sample arbitrary discrete probability distributions p_k in O(log_2 K) steps by constructing a cumulative probability table c_k and applying binary bisection search.
- Derive analytic inverse-CDF mappings x = C^{-1}(u) for exponential waiting times, power-law distributions, and Poisson decay counts.
- Implement von Neumann rejection sampling with proposal envelope a q(x) >= p(x), prove that the acceptance efficiency equals 1/a, and explain its exponential collapse in high dimensions.
- Generate 1D normal variates via the 12-uniform Central Limit composition trick, trigonometric Box-Muller, and rejection-based Polar Box-Muller, and transform them into D-dimensional correlated Gaussians via Cholesky factorization S S^T = T.
- Optimize Monte Carlo efficiency zeta = 1/(v T) via Importance Sampling, derive the zero-variance optimal density p*(x) = |f(x)| / int |f|, and diagnose the infinite-variance tail trap (a <= 1/2) in Gaussian importance sampling.
- Formulate correlated sampling for free-energy or parameter differences and prove why independent uniform sampling of the canonical partition function Z(beta) fails exponentially with particle count N.


In [2]:
ws.render_section('sec_discrete_and_inverse_mapping')

## 1. Transforming Uniform Draws: Discrete Bisection Tables & Continuous Inverse Mapping

In Lesson 13 we learned how to generate a stream of independent **uniform random numbers** (`udrn`) $u \sim U(0, 1)$. To simulate physical processes (such as Kinetic Monte Carlo event selection, Maxwell–Boltzmann velocities, or radioactive decay) or to reduce integration variance, we must transform $u \in (0, 1)$ into draws from an arbitrary **non-uniform** distribution $p(x)$.

### Discrete Distributions: Cumulative Table & Bisection Search
Suppose an system can jump into one of $K$ discrete outcomes with normalized probabilities $p_1, p_2, \dots, p_K$ ($\sum_{i=1}^K p_i = 1$).
1. **Precompute the Cumulative Table** before the simulation loop ($c_0 = 0$, $c_K = 1$):
 $$c_k = \sum_{i=1}^k p_i = c_{k-1} + p_k \qquad (k = 1, 2, \dots, K)$$
2. **Sample $u \in (0, 1)$** and locate the unique bin index $k$ satisfying:
 $$c_{k-1} < u \le c_k$$
3. **Search Complexity:** Because $0 = c_0 \le c_1 \le \dots \le c_K = 1$ is monotonically sorted, we locate $k$ by **bisection (binary search)** in **$\mathcal{O}(\log_2 K)$** steps instead of an $\mathcal{O}(K)$ linear scan!

### Continuous Distributions: The Inverse Mapping Method
Generalizing the cumulative sum to a continuous probability density $p(x)$ ($\int_{-\infty}^{\infty} p(x)\,dx = 1$), we define the **Cumulative Distribution Function (CDF)** $C(y)$ as the area under $p(x)$ to the left of $y$:
$$C(y) = \int_{-\infty}^y p(x)\,dx \in [0, 1]$$
Setting $C(x) = u \sim U(0, 1)$ and inverting gives the **exact inverse mapping**:
$$x = C^{-1}(u)$$
*(Why does this work? Because $du = dC(x) = p(x)\,dx$, a uniform density $du$ on $(0,1)$ maps directly onto the probability element $p(x)\,dx$!)*

#### Two Essential Analytic Inverse Mappings
1. **Exponential Distribution (KMC Escape Times / Radioactive Decay):**
 $$p(x) = a e^{-ax} \quad (x > 0, \; a > 0) \implies C(x) = \int_0^x a e^{-ay}\,dy = 1 - e^{-ax} = u$$
 Solving for $x$ (and noting that $1 - u$ has the exact same uniform distribution on $(0,1)$ as $u$):
 $$x = -\frac{\ln(1 - u)}{a} \equiv -\frac{\ln(u)}{a}$$
2. **Power-Law Distribution on $[0, 1]$:**
 $$p(x) = (a + 1)x^a \quad (0 < x < 1, \; a > -1) \implies C(x) = \int_0^x (a+1)y^a\,dy = x^{a+1} = u \implies x = u^{1/(a+1)}$$

### Connection to the Poisson Distribution
In $N$ independent Bernoulli trials each with small success probability $p = \nu / N$ (expected successes $\nu = Np$), the binomial distribution $P_B(n|N) = \frac{N!}{n!(N-n)!}p^n(1-p)^{N-n}$ converges as $N \to \infty$ to the **Poisson distribution**:
$$P_{\text{Poisson}}(n) = \lim_{N\to\infty} \frac{N!}{n!(N-n)!}\left(\frac{\nu}{N}\right)^n\left(1 - \frac{\nu}{N}\right)^{N-n} = \frac{\nu^n e^{-\nu}}{n!}$$
which peaks at $n = \nu$. Because the waiting time between consecutive Poisson events at rate $a$ is exponentially distributed ($y_k = -\ln(u_k)/a$), the number of events $n$ occurring within time interval $t$ ($\nu = at$) can be sampled either by inverting the discrete cumulative table $F(n) = \sum_{m=0}^n \frac{\nu^m e^{-\nu}}{m!}$ or by accumulating exponential waiting times until $\sum_{k=1}^{n+1} y_k > t$ (equivalently, multiplying uniform draws until $\prod_{k=1}^{n+1} u_k < e^{-at}$).


In [3]:
ws.render_section('q14_01_discrete_cumulative_table')

**Q1 · easy** · 💡

A Kinetic Monte Carlo site has $K = 4$ possible hopping events with normalized probabilities $p_1 = 0.15$, $p_2 = 0.35$, $p_3 = 0.30$, and $p_4 = 0.20$. You construct the cumulative table $c_k = \sum_{i=1}^k p_i$ ($c_1 = 0.15, c_2 = 0.50, c_3 = 0.80, c_4 = 1.00$) and draw a uniform random number $u = 0.62$. Which event index $k \in \{1, 2, 3, 4\}$ is selected by the rule $c_{k-1} < u \le c_k$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Find the interval $(c_{k-1}, c_k]$ that brackets $u = 0.62$.

</div>
</details>

_Numeric answer required._

In [4]:
ws.render_section('q14_02_bisection_search_steps')

**Q2 · easy** · 💡

In a large lattice Kinetic Monte Carlo simulation, your cumulative probability table has $N = 1024$ possible discrete events. How many comparison steps are required to locate the selected event $c_{k-1} < u \le c_k$ using **bisection (binary search)**, compared to $1024$ worst-case steps for a naive linear scan?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the search interval is halved at each step of a binary search until a single element remains.

</div>
</details>

_Numeric answer required._

In [5]:
ws.render_section('q14_03_exponential_inverse_mapping')

**Q3 · easy** · 💡

You are sampling an exponential waiting-time distribution $p(x) = a e^{-ax}$ ($x > 0$) with rate parameter $a = 2.0\text{ ps}^{-1}$ via inverse transform sampling. If the uniform PRNG returns $u = e^{-3} \approx 0.049787$, what waiting time $x$ (in $\text{ps}$) is generated?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Invert the cumulative distribution function of the exponential distribution and evaluate at the given deviate.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q14_04_power_law_inverse_mapping')

**Q4 · medium** · 💡

To sample the normalized power-law distribution $p(x) = (a + 1)x^a$ on $0 < x < 1$ with exponent $a = 2$ (so $p(x) = 3x^2$), you apply inverse transform sampling. If your uniform draw is $u = 0.125$, what value of $x$ is returned?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the power-law density to find the CDF on $[0, 1]$, invert for $x(u)$, and evaluate at the given uniform draw.

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q14_05_custom_linear_pdf_inversion')

**Q5 · medium** · 💡

Suppose you want to sample a radial coordinate $r \in [0, R]$ in 2D with probability density proportional to circumference, $p(r) = \frac{2r}{R^2}$ for $0 \le r \le R$. What is the exact inverse mapping $r(u)$ in terms of a uniform random number $u \in (0, 1)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Find the cumulative distribution function by integrating $p(r)$ from 0 to $r$, then equate to $u$ and invert.

</div>
</details>

- [ ] **(A)** $r = R\sqrt{u}$
- [ ] **(B)** $r = R u^2$
- [ ] **(C)** $r = -R \ln(u)$
- [ ] **(D)** $r = R u / 2$

In [8]:
ws.render_section('q14_06_poisson_limit_prob')

**Q6 · medium** · 💡

In the Poisson limit $P(n) = \frac{\nu^n e^{-\nu}}{n!}$ of $N \to \infty$ trials with expected successes $\nu = Np = 2.0$, what is the exact probability $P(n = 2)$ of observing $2$ events? (Enter as a decimal to 3 decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the discrete Poisson distribution formula using the given mean event count.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q14_07_inverse_mapping_limitation')

**Q7 · easy** · 💡

When is the direct continuous inverse mapping method $x = C^{-1}(u)$ **impractical** to use by itself?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about high-dimensional distributions or 1D PDFs whose cumulative integral $C(y) = \int_{-\infty}^y p(x)dx$ cannot be inverted in closed form.

</div>
</details>

- [ ] **(A)** When the cumulative integral $C(y)$ or its inverse $C^{-1}(u)$ has no closed-form analytic expression (or is too slow to evaluate numerically) and the problem is multi-dimensional.
- [ ] **(B)** Whenever the probability density $p(x)$ is positive ($p(x) > 0$).
- [ ] **(C)** Whenever $p(x)$ is an exponential distribution $a e^{-ax}$.
- [ ] **(D)** Whenever the integral of $p(x)$ over all space equals $1$.

In [10]:
ws.render_section('sec_rejection_composition_and_normals')

### The von Neumann Rejection Technique
Suppose we want to sample from a normalized target density $p(x)\,dx$, and we have a simpler **proposal (comparison) distribution** $q(x)\,dx$ that we already know how to sample.
1. Choose an envelope constant $a \ge 1$ large enough that $a\,q(x) \ge p(x)$ everywhere, so the **acceptance probability** $c(x)$ satisfies:
 $$c(x) = \frac{p(x)}{a\,q(x)} \le 1 \quad \forall x \qquad \left(\text{i.e., } a \ge \max_x \frac{p(x)}{q(x)}\right)$$
 *(Note: of the lecture writes $c(x) = a\,p(x)/q(x) \le 1$ with $a \le \min[q/p]$ and then denotes the acceptance efficiency by $1/a$; both conventions express the exact same condition: the proposal envelope must bound $p(x)$ from above, and the overall fraction of accepted proposals is the ratio of the area under $p(x)$ to the area under the envelope!)*
2. **Algorithm Loop:**
 - Draw a candidate $x \sim q(x)\,dx$.
 - Draw a uniform random number $u \sim U(0, 1)$.
 - If $u \le c(x)$, **accept** $x$; otherwise **reject** $x$ and repeat.
3. **Proof & Efficiency:** The probability that a proposed $x$ lies in $[x, x+dx]$ *and* is accepted is $q(x)\,dx \cdot c(x) = \frac{1}{a}p(x)\,dx$. Integrating over all $x$ shows that the **overall acceptance probability (efficiency)** is:
 $$\text{Efficiency} = \int_{-\infty}^{\infty} q(x) c(x)\,dx = \frac{1}{a}\int_{-\infty}^{\infty} p(x)\,dx = \frac{1}{a}$$
 - **Rule:** Choose $q(x)$ to match the shape of $p(x)$ as closely as possible so $a$ is close to $1$. In $D$ dimensions, rejection efficiency scales as $(1/a)^D$, collapsing exponentially for large $D$!

### Composition Methods (Combining Uniform Draws)
1. **Adding Uniforms (Central Limit Theorem Trick):** A single uniform $u_i \in [0, 1]$ has mean $1/2$ and variance $1/12$. By the Central Limit Theorem, summing $12$ independent uniforms and subtracting $6$:
 $$\zeta = \sum_{i=1}^{12} u_i - 6$$
 gives a fast, approximate **standard Gaussian** ($\langle \zeta \rangle = 0$, $\text{Var}(\zeta) = 12 \times \frac{1}{12} = 1$) supported on $(-6, 6)$! *(Warning: Because it is strictly bounded on $[-6, 6]$, never use the 12-udrn trick when rare $>4\sigma$ tail events matter!)*
2. **Maximum of $k$ Uniforms:** Let $x = \max(u_1, u_2, \dots, u_k)$ where $u_i \sim U(0, 1)$ are independent. The probability that all $k$ numbers are $\le x$ is the CDF $C(x) = \prod_{i=1}^k \text{Prob}(u_i \le x) = x^k$. Differentiating gives the exact PDF:
 $$P(x) = \frac{d}{dx}\left(x^k\right) = k\,x^{k-1} \quad (0 \le x \le 1)$$

### Exact 1D Normal Distribution: Box–Muller & Polar Rejection
Instead of inverting $\text{erf}(y)$ for a single Gaussian $p(y) = (2\pi)^{-1/2}e^{-y^2/2}$, we generate **two** independent standard normals $(y_1, y_2)$ at once in 2D polar coordinates $(r, \theta)$:
$$p(y_1, y_2)\,dy_1\,dy_2 = \frac{1}{2\pi}e^{-(y_1^2 + y_2^2)/2}\,dy_1\,dy_2 = \left(\frac{d\theta}{2\pi}\right)\left(\frac{1}{2}e^{-v/2}\,dv\right), \quad v = r^2$$
Since $\theta$ is uniform on $[0, 2\pi)$ and $v = r^2$ is exponentially distributed with rate $a = 1/2$ ($v = -2\ln x_1$), drawing $x_1, x_2 \sim U(0, 1)$ yields the **Box–Muller transform**:
$$y_1 = \sqrt{-2\ln x_1}\cos(2\pi x_2), \qquad y_2 = \sqrt{-2\ln x_1}\sin(2\pi x_2)$$
- **Polar Rejection Box–Muller (No Trig Functions!):** As shown in the lecture's Python code, we eliminate slow $\sin/\cos$ calls by sampling $(x, y) = (\text{udrn}-0.5, \text{udrn}-0.5)$ inside the unit square $[-0.5, 0.5]^2$ and **rejecting** until $r^2 = x^2 + y^2 \le 0.25$ (inscribed circle of radius $1/2$, area $\pi/4$ inside square of area $1$). Then $x/\sqrt{r^2} = \cos\theta$ and $y/\sqrt{r^2} = \sin\theta$, so setting `radius = sigma * sqrt(-2 * ln(udrn) / r2)` produces two exact normal variates `x0 + x * radius` and `y0 + y * radius`!

### Multivariate Correlated Gaussians via Cholesky Decomposition
How do we sample a $D$-component Gaussian vector $\mathbf{x} = (x_1, \dots, x_D)^T$ with mean $\mathbf{0}$ and target symmetric positive-definite covariance matrix $\langle x_\alpha x_\beta \rangle = T_{\alpha\beta}$?
1. Compute the **Cholesky ("square-root") decomposition** of $\mathbf{T}$:
 $$\mathbf{S}\mathbf{S}^T = \mathbf{T}$$
 where $\mathbf{S}$ is a **lower-triangular matrix** ($S_{\alpha i} = 0$ for all entries above the diagonal, $i > \alpha$).
2. Generate $D$ independent standard normal variates $\mathbf{y} = (y_1, \dots, y_D)^T$ with $\langle y_i y_j \rangle = \delta_{ij}$.
3. Apply the linear transformation $\mathbf{x} = \mathbf{S}\mathbf{y}$ (or $\mathbf{x} = \boldsymbol{\mu} + \mathbf{S}\mathbf{y}$ for nonzero mean):
 $$x_\alpha = \sum_{i=1}^\alpha S_{\alpha i} y_i \implies \langle x_\alpha x_\beta \rangle = \sum_{i,j} S_{\alpha i} S_{\beta j} \underbrace{\langle y_i y_j \rangle}_{\delta_{ij}} = \sum_i S_{\alpha i} S_{\beta i} = (\mathbf{S}\mathbf{S}^T)_{\alpha\beta} = T_{\alpha\beta}$$


In [11]:
ws.render_section('concept_rejection_sampling')

### Interactive Lab: von Neumann Rejection Sampling & Envelope Efficiency

Explore von Neumann rejection sampling on $[0, 1]$ using a uniform proposal $q(x) = 1$. Adjust the **Envelope Slack Factor** ($a / a_{\min} \ge 1$) and switch between target distributions $p(x)$ to observe how excess area between the red dashed envelope $a\,q(x)$ and the green target density $p(x)$ turns directly into rejected Monte Carlo proposals ($1/a$ efficiency).

**Key Takeaway:** Every point below $p(x)$ (green) is accepted ($u \le p(x)/[a\,q(x)]$), while every point between $p(x)$ and $a\,q(x)$ (red) is rejected. The fraction of accepted proposals converges to the area ratio $\int p(x)dx / \int a\,q(x)dx = 1/a$. Making $a$ larger than necessary wastes random numbers without changing the accepted distribution, while setting $a < \max[p/q]$ clips the peaks and biases the sample!

In [12]:
ws.render_section('concept_cholesky_gaussian')

### Interactive Lab: Multivariate Correlated Gaussians via Cholesky Factorization

Tune the target standard deviations $(\sigma_1, \sigma_2)$ and correlation coefficient $\rho \in (-1, 1)$ of a 2D covariance matrix $\mathbf{T} = \begin{pmatrix} \sigma_1^2 & \rho\sigma_1\sigma_2 \\ \rho\sigma_1\sigma_2 & \sigma_2^2 \end{pmatrix}$. Watch how the lower-triangular Cholesky factor $\mathbf{S}$ ($\mathbf{S}\mathbf{S}^T = \mathbf{T}$) scales and shears an isotropic normal point cloud $\mathbf{y} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ into the exact correlated Gaussian ellipse.

**Key Takeaway:** Because $\mathbf{S}$ is lower-triangular ($S_{12} = 0$), $x_1 = S_{11} y_1 = \sigma_1 y_1$ depends only on $y_1$, while $x_2 = S_{21} y_1 + S_{22} y_2 = \sigma_2(\rho y_1 + \sqrt{1-\rho^2}\,y_2)$ mixes $y_1$ and $y_2$ to produce the exact cross-correlation $\langle x_1 x_2 \rangle = \rho\sigma_1\sigma_2$.

In [13]:
ws.render_section('q14_08_rejection_acceptance_prob')

**Q8 · easy** · 💡

You use von Neumann rejection sampling to draw from a target density $p(x)$ on $[0, 1]$ using a uniform proposal density $q(x) = 1$. If the maximum value of $p(x)$ on $[0, 1]$ is $p_{\max} = 2.5$ and you choose the tightest valid envelope multiplier $a = 2.5$ so that $c(x) = p(x) / [a\,q(x)] \le 1$, what is the exact overall **acceptance efficiency** of the sampler? (Enter as a decimal.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the overall acceptance efficiency of von Neumann rejection sampling to the bounding envelope scale factor.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q14_09_rejection_semicircle_envelope')

**Q9 · medium** · 💡

Consider sampling the normalized Wigner semicircle distribution $p(x) = \frac{8}{\pi}\sqrt{x(1-x)}$ on $[0, 1]$ using a uniform proposal $q(x) = 1$ (the first preset in the Rejection Sampling lab above). What is the minimum valid envelope constant $a_{\min} = \max_{x \in [0,1]} [p(x) / q(x)]$ so that $c(x) \le 1$ for all $x \in [0, 1]$? (Enter to 3 decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Find the maximum value of the semicircle target distribution on the interval by evaluating the peak of $x(1-x)$.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q14_10_rejection_high_dim_curse')

**Q10 · medium** · 💡

Suppose a 1D rejection sampler has an acceptance efficiency of $50\%$. If you apply the same independent factorized proposal envelope across $D = 10$ dimensions, what is the $10$-dimensional acceptance probability, expressed as a **percentage**? (Enter the number of percent, e.g. Enter `0.0977` for $0.0977\%$.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply the independent per-dimension acceptance probabilities across all 10 spatial dimensions and express as a percentage.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q14_11_composition_12_udrn')

**Q11 · easy** · 💡

Why does the classic composition formula $\zeta = \sum_{i=1}^{12} u_i - 6$ (where $u_1, \dots, u_{12}$ are independent uniform random numbers on $[0, 1]$) specifically sum **$12$** uniform numbers and subtract **$6$**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that a single uniform draw $u_i \in [0, 1]$ has mean $1/2$ and variance $1/12$.

</div>
</details>

- [ ] **(A)** Because each $u_i$ has mean $1/2$ and variance $1/12$, so summing $12$ independent uniforms produces mean $12 \times (1/2) = 6$ (shifted to $0$ by subtracting $6$) and unit variance $12 \times (1/12) = 1$ without requiring a square-root scaling.
- [ ] **(B)** Because $12$ is the minimum integer for which $\zeta$ has infinite tails $(-\infty, \infty)$.
- [ ] **(C)** Because a 12-bit LCG has a period of $6$.
- [ ] **(D)** Because $\sum_{i=1}^{12} u_i - 6$ reproduces the exact Box–Muller cosine transform.

In [17]:
ws.render_section('q14_12_max_of_k_uniforms_pdf')

**Q12 · medium** · 💡

If you draw $k$ independent uniform random numbers $u_1, u_2, \dots, u_k \in [0, 1]$ and take their maximum, $x = \max(u_1, u_2, \dots, u_k)$, what is the probability density function $P(x)$ of $x$ on $[0, 1]$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

The event $\max(u_1, \dots, u_k) \le x$ happens if and only if ALL $k$ independent draws satisfy $u_i \le x$, so the CDF is $C(x) = x^k$. Differentiate $C(x)$!

</div>
</details>

- [ ] **(A)** $P(x) = k\,x^{k-1}$
- [ ] **(B)** $P(x) = x^k / k!$
- [ ] **(C)** $P(x) = k(1 - x)^{k-1}$
- [ ] **(D)** $P(x) = -\ln(x) / k$

In [18]:
ws.render_section('q14_13_max_of_3_uniforms_mean')

**Q13 · medium** · 💡

Using the composition rule $x = \max(u_1, u_2, u_3)$ ($k = 3$, which has PDF $P(x) = 3x^2$ on $[0, 1]$), what is the exact expected value $\langle x \rangle = \int_0^1 x P(x)\,dx$? (Enter as a decimal.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate $x P(x)$ over the unit interval using the PDF of the maximum of three uniforms.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q14_14_box_muller_radial_distribution')

**Q14 · medium** · 💡

In the 2D Box–Muller derivation for two independent standard normals $(y_1, y_2)$, transforming $p(y_1, y_2)\,dy_1\,dy_2 = \frac{1}{2\pi}\exp(-r^2/2)\,r\,dr\,d\theta$ to the squared radius $v = r^2$ ($dv = 2r\,dr$) yields which probability distribution for $v$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes: $p(v)dv = \frac{1}{2}\exp(-v/2)dv$.

</div>
</details>

- [ ] **(A)** An exponential distribution $p(v) = \frac{1}{2}e^{-v/2}$ ($v \ge 0$) with rate parameter $a = 1/2$, which can be sampled analytically via $v = r^2 = -2\ln(x_1)$.
- [ ] **(B)** A uniform distribution $p(v) = 1$ on $[0, 1]$, sampled via $v = x_1$.
- [ ] **(C)** A Cauchy distribution $p(v) = \frac{1}{\pi(1 + v^2)}$.
- [ ] **(D)** A Poisson distribution with discrete integer radii $v \in \{0, 1, 2, \dots\}$.

In [20]:
ws.render_section('q14_15_box_muller_polar_rejection_efficiency')

**Q15 · medium** · 💡

In the lecture's **Polar Box–Muller code** - 0.5, sprng - 0.5
 r2 = x*x + y*y
```
points $(x, y)$ are sampled uniformly inside the $1 \times 1$ square $[-0.5, 0.5]^2$ and accepted if they fall inside the circle of radius $R = 0.5$ ($r^2 \le 0.25$). What is the probability $p_{\text{accept}}$ that a single $(x, y)$ pair is accepted on any given iteration of the `while` loop? (Enter as a decimal to 3 decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute the ratio of the area of the circular acceptance disk to the area of the bounding square domain.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q14_16_box_muller_polar_advantages')

**Q16 · easy** · 💡

According to of the reference notes, what are the computational and statistical advantages of the rejection-based **Polar Box–Muller** algorithm? Select **all** true statements:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the bullet points at the bottom of.

</div>
</details>

- [ ] **(A)** It completely avoids calling slow trigonometric functions (`sin` and `cos`).
- [ ] **(B)** The rejection step consumes a variable number of random numbers, which mixes up the crystal-plane regularity (Marsaglia hyperplanes) of simple LCGs.
- [ ] **(C)** Each successful pass generates $2$ independent normal random numbers (`xnormal` and `ynormal`) at once.
- [ ] **(D)** It never calls `sqrt` or `ln`.

In [22]:
ws.render_section('q14_17_cholesky_matrix_equation')

**Q17 · easy** · 💡

To generate a $D$-dimensional correlated Gaussian vector $\mathbf{x}$ with target covariance matrix $T_{\alpha\beta} = \langle x_\alpha x_\beta \rangle$ from an independent standard normal vector $\mathbf{y}$ ($\langle y_i y_j \rangle = \delta_{ij}$) via $\mathbf{x} = \mathbf{S}\mathbf{y}$, which matrix equation must the lower-triangular Cholesky matrix $\mathbf{S}$ satisfy?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $x_\alpha = \sum_i S_{\alpha i} y_i$ into $\langle x_\alpha x_\beta \rangle = T_{\alpha\beta}$.

</div>
</details>

- [ ] **(A)** $\mathbf{S}\mathbf{S}^T = \mathbf{T}$
- [ ] **(B)** $\mathbf{S} + \mathbf{S}^T = \mathbf{T}^{-1}$
- [ ] **(C)** $\mathbf{S}^2 = \mathbf{I}$
- [ ] **(D)** $\det(\mathbf{S}) = 0$

In [23]:
ws.render_section('q14_18_cholesky_2x2_entry_s22')

**Q18 · hard** · 💡

Suppose you want to generate a 2D correlated Gaussian $(x_1, x_2)$ with covariance matrix:
$$\mathbf{T} = \begin{pmatrix} 4.0 & 2.4 \\ 2.4 & 2.25 \end{pmatrix}$$
($\sigma_1 = 2.0$, $\sigma_2 = 1.5$, $\rho = 0.8$). You write the lower-triangular Cholesky factor as $\mathbf{S} = \begin{pmatrix} S_{11} & 0 \\ S_{21} & S_{22} \end{pmatrix}$ with $S_{11}, S_{22} > 0$. From $\mathbf{S}\mathbf{S}^T = \mathbf{T}$, we have $S_{11}^2 = 4.0 \implies S_{11} = 2.0$ and $S_{11}S_{21} = 2.4 \implies S_{21} = 1.2$. What is the value of the diagonal entry **$S_{22}$**? (You can also check this by setting $\sigma_1=2.0, \sigma_2=1.5$ in the Cholesky lab!)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate the $(2,2)$ element of the matrix product $\mathbf{S}\mathbf{S}^T$ to $T_{22}$ and solve for $S_{22}$.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q14_19_cholesky_positive_definite')

**Q19 · medium** · 💡

What mathematical property must a real symmetric matrix $\mathbf{T}$ possess in order to represent a valid physical covariance matrix and admit a real Cholesky factorization $\mathbf{S}\mathbf{S}^T = \mathbf{T}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

For any non-zero vector $\mathbf{c}$, the variance of the linear combination $\mathbf{c}^T \mathbf{x}$ is $\mathbf{c}^T \mathbf{T} \mathbf{c} > 0$.

</div>
</details>

- [ ] **(A)** $\mathbf{T}$ must be positive-definite (all eigenvalues $> 0$, or $|ho| < 1$ with $\sigma_i > 0$), so that $\mathbf{c}^T \mathbf{T} \mathbf{c} > 0$ for all non-zero vectors $\mathbf{c}$.
- [ ] **(B)** $\mathbf{T}$ must be antisymmetric ($T_{ij} = -T_{ji}$) with zeros on the diagonal.
- [ ] **(C)** $\mathbf{T}$ must have negative diagonal entries ($T_{ii} < 0$).
- [ ] **(D)** $\mathbf{T}$ must be an orthogonal permutation matrix.

In [25]:
ws.render_section('sec_importance_sampling_and_variance_trap')

### The Figure of Merit: Simulation Efficiency $\zeta$
Because the statistical error of a Monte Carlo mean decays as $\text{error} = \sqrt{\text{variance} / N} \propto (\text{computer time})^{-1/2}$, the product of the **squared error of the mean** $v = \text{error}^2 = \sigma^2 / N$ and the **total CPU time** $T = N \tau_{\text{step}}$ is independent of run length $N$! We therefore define the fundamental **Efficiency (Figure of Merit)** of a Monte Carlo algorithm as:
$$\text{Efficiency} = \zeta = \frac{1}{v\,T}$$
To maximize $\zeta$, we can write a faster code (smaller $\tau_{\text{step}}$), buy a faster computer, or **reduce the per-sample variance $\sigma^2$** via **Importance Sampling** and **Correlated Sampling**.

### Importance Sampling & The Zero-Variance Principle
Suppose we want to evaluate the multidimensional integral $I = \int dx\,f(x)$. Instead of sampling $x$ uniformly, we introduce a normalized **importance sampling probability density** $p(x) > 0$ ($\int dx\,p(x) = 1$):
$$I = \int dx\,p(x)\left(\frac{f(x)}{p(x)}\right) = \left\langle \frac{f(x)}{p(x)} \right\rangle_p$$
Drawing $N$ samples $x_i \sim p(x)\,dx$, our unbiased **estimator** is $g(x) = f(x)/p(x)$. Its **mean** is always $I$ (independent of $p(x)$!), but its **variance** depends strongly on $p(x)$:
$$v \approx \sigma^2 = \left\langle \left(\frac{f(x)}{p(x)} - I\right)^2 \right\rangle_p = \int dx\,p(x)\left(\frac{f(x)}{p(x)}\right)^2 - I^2 = \int dx\,\frac{f(x)^2}{p(x)} - I^2$$
Minimizing $\sigma^2$ with respect to $p(x)$ subject to the normalization constraint $\int dx\,p(x) = 1$ (e.g., via Lagrange multipliers or parameterizing $p(x) = [q(x)]^2 / \int [q]^2$) yields the **Optimal Importance Function**:
$$p^*(x) = \frac{|f(x)|}{\int dx'\,|f(x')|}, \qquad \text{Estimator: } \frac{f(x)}{p^*(x)} = \text{sign}\big(f(x)\big)\int dx'\,|f(x')|$$
1. **Zero-Variance Principle:** If $f(x)$ does not change sign ($f(x) \ge 0$ everywhere), the estimator $f(x)/p^*(x) = I$ is a **constant**, so its variance is **identically zero** ($\sigma^2 = 0$)!
2. **Practical Guide:** Of course, we cannot normalize $p^*(x)$ directly without already knowing $I = \int |f(x)|dx$—if we could, we would have solved the problem analytically! Instead, we choose a solvable trial function $p_a(x)$ whose shape mimics $|f(x)|$ where $|f(x)|$ is large.
3. **The Sign Problem:** If $f(x)$ oscillates between positive and negative values, $\text{sign}(f(x))$ jumps between $+1$ and $-1$, so **zero variance is impossible** even with $p^*(x)$! As the lecture notes: *"Monte Carlo can add, but not subtract."*

### Lecture Case Study: Gaussian Importance Sampling & The Infinite-Variance Trap
Consider the lecture's classic 1D integral = \frac{e^{-x^2/2}}{1 + x^2}, \qquad \text{sampled with trial Gaussian } p_a(x) = \frac{e^{-x^2/(2a)}}{\sqrt{2\pi a}} \quad (a > 0)$$
- **Unbiased Mean:** For any $a > 0$, $\langle f(x)/p_a(x) \rangle_{p_a} = \int dx\,f(x) = I \approx 1.6436$ is independent of $a$.
- **Variance Integral:** Substituting $f(x)$ and $p_a(x)$ into $\sigma^2(a) = \int dx\,\frac{f(x)^2}{p_a(x)} - I^2$ gives:
 $$\sigma^2(a) = \int_{-\infty}^{\infty} dx\,\frac{\sqrt{2\pi a}}{(1 + x^2)^2}\,\exp\!\left[-x^2\left(1 - \frac{1}{2a}\right)\right] - I^2$$
- **Condition for Finite Variance ($a > 1/2$):** Look at the Gaussian exponent $-x^2\left(1 - \frac{1}{2a}\right)$ as $|x| \to \infty$:
 - If **$a \le 1/2$** ($1 - \frac{1}{2a} \le 0$), the exponential **blows up** as $|x| \to \infty$! The variance integral diverges ($\sigma^2(a) = \infty$). In a simulation, drawing a rare point in the tail $|x| \gg 1$ where $p_a(x) \ll f(x)$ produces gigantic **Pareto spikes** ($f(x_i)/p_a(x_i) \gg 1$) and long power-law tails that ruin convergence!
 - If **$a = 1$** (the "obvious" choice matching the numerator $e^{-x^2/2}$), variance is finite ($\sigma^2(1) \approx 0.17$).
 - **Optimal Value ($a^* \approx 0.6$):** Because the denominator $1 + x^2$ makes $f(x)$ narrower than $e^{-x^2/2}$ near the origin, narrowing the Gaussian slightly to **$a \approx 0.6$** minimizes $\sigma^2(a)$ and cuts CPU time by more than a factor of $6$ compared to $a = 1$!
 -*(Note: If the integrand were a pure Gaussian $f(x) = e^{-x^2/2}$ without the $1/(1+x^2)$ factor, the exact same exponent $1 - 1/(2a) > 0$ requires $a > 1/2$ for finite variance, while $a = 1$ would give zero variance! And if one tried to evaluate the unweighted average $\langle 1/p_a(x) \rangle$, the integral would diverge unless $p_a$ were wider still; notes the three regimes: $a > 0$ for $p(x)$ to exist, $a > 0.5$ for finite variance, $a = 0.6$ optimal, and $a = 1$ obvious!)*

> [!CAUTION]
> **Golden Rule of Importance Sampling:** **Never under-sample the tails!** Over-sampling the tails ($a > 0.6$, broader $p(x)$) causes only a mild, smooth loss of efficiency, whereas under-sampling the tails ($a \le 0.5$, narrower $p(x)$ than $\sqrt{|f(x)|}$) causes **infinite variance** and catastrophic spikes!


In [26]:
ws.render_section('concept_importance_sampling_trap')

### Interactive Lab: The Gaussian Importance Sampling Variance Trap

Reproduce the exact numerical experiment of the reference notes: integrating $f(x) = \frac{e^{-x^2/2}}{1+x^2}$ using the trial Gaussian $p_a(x) = \frac{e^{-x^2/(2a)}}{\sqrt{2\pi a}}$. Drag the Gaussian variance parameter **$a$** below the critical threshold **$a = 0.50$** (e.g., $a = 0.20$ or $0.40$) to see the violent **infinite-variance spikes**, then tune $a$ to the **optimal value $a \approx 0.60$** and switch to the **Theoretical Variance / CPU Time Curve**!

**Key Takeaway:** Compare $a = 0.60$ (tight, symmetric fluctuations around $I = 1.6436$ with tiny error bars) against $a = 0.25$ or $0.40$ (deceptively quiet baseline punctuated by massive upward spikes whenever a sample lands in the tail $|x| > 2$ where $p_a(x)$ is exponentially smaller than $f(x)$). Always check analytically that $\int [f(x)^2 / p(x)]\,dx < \infty$ before trusting an importance sampler!

In [27]:
ws.render_section('q14_20_mc_efficiency_definition')

**Q20 · easy** · 💡

How is the **Efficiency (figure of merit)** $\zeta$ of a Monte Carlo simulation defined in terms of the squared statistical error of the mean $v = \text{error}^2$ and the total CPU time $T$, and why is $\zeta$ independent of the total number of Monte Carlo steps $N$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the yellow box of the reference notes: $\zeta = 1 / (v T)$.

</div>
</details>

- [ ] **(A)** $\zeta = \frac{1}{v T}$; because $v = \sigma^2 / N \propto N^{-1}$ and CPU time $T = N \tau_{\text{step}} \propto N$, the factors of $N$ cancel out in $v T = \sigma^2 \tau_{\text{step}}$.
- [ ] **(B)** $\zeta = v T$; because both variance $v$ and CPU time $T$ decrease linearly with $N$.
- [ ] **(C)** $\zeta = \frac{T}{v}$; because longer CPU times always imply higher efficiency.
- [ ] **(D)** $\zeta = \sqrt{v / T}$; which grows as $N^2$.

In [28]:
ws.render_section('q14_21_efficiency_improvement_calculation')

**Q21 · medium** · 💡

Suppose introducing an importance sampling function $p(x)$ reduces the per-sample estimator variance $\sigma^2$ by a factor of $10$ (from $\sigma_0^2$ to $\sigma_0^2 / 10$), while evaluating $p(x)$ increases the CPU time per step $\tau_{\text{step}}$ by a factor of $2$. By what factor $\zeta_{\text{new}} / \zeta_{\text{old}}$ does the overall simulation **efficiency** $\zeta = 1/(v T)$ improve?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the ratio of the efficiency figures of merit using the variance reduction factor and the per-step CPU time increase.

</div>
</details>

_Numeric answer required._

In [29]:
ws.render_section('q14_22_importance_sampling_estimator')

**Q22 · easy** · 💡

When evaluating $I = \int dx\,f(x)$ by drawing points $x_i$ from a normalized non-uniform probability distribution $p(x)$ ($\int dx\,p(x) = 1$), what is the unbiased Monte Carlo estimator $\hat{I}$ for $N$ samples, and how does its expected value $\langle \hat{I} \rangle_p$ depend on the choice of $p(x)$ (assuming $p(x) > 0$ wherever $f(x) \ne 0$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes: 'Mean value of estimator I is independent of p(x), but variance v is not!'

</div>
</details>

- [ ] **(A)** The estimator is $\hat{I} = \frac{1}{N}\sum_{i=1}^N \frac{f(x_i)}{p(x_i)}$; its expected mean value $\langle \hat{I} \rangle_p = I$ is completely independent of $p(x)$, whereas its variance depends strongly on $p(x)$.
- [ ] **(B)** The estimator is $\hat{I} = \frac{1}{N}\sum_{i=1}^N f(x_i) p(x_i)$; both its mean and variance are independent of $p(x)$.
- [ ] **(C)** The estimator is $\hat{I} = \frac{1}{N}\sum_{i=1}^N f(x_i)$; its mean equals $I$ only when $p(x)$ is a Gaussian.
- [ ] **(D)** The estimator is $\hat{I} = \frac{1}{N}\sum_{i=1}^N \frac{p(x_i)}{f(x_i)}$; its variance is always zero for any $p(x)$.

In [30]:
ws.render_section('q14_23_optimal_p_star_formula')

**Q23 · medium** · 💡

By functionally minimizing the importance sampling variance $v[p] = \int dx\,p(x)\left(\frac{f(x)}{p(x)}\right)^2 - I^2$ subject to $\int dx\,p(x) = 1$, what is the **optimal importance sampling distribution** $p^*(x)$, and why can we not simply use the exact $p^*(x)$ directly in every simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review of the reference notes ('Finding Optimal p*(x) for Sampling').

</div>
</details>

- [ ] **(A)** $p^*(x) = \frac{|f(x)|}{\int dx'\,|f(x')|}$; we cannot sample $p^*(x)$ directly in general because its normalization constant $\int dx'\,|f(x')|$ IS the very integral we are trying to compute!
- [ ] **(B)** $p^*(x) = \frac{1}{f(x)^2}$; we cannot use it because $f(x)^2$ is always negative.
- [ ] **(C)** $p^*(x) = \delta(x - x_0)$; we cannot use it because computers cannot store floating-point numbers.
- [ ] **(D)** $p^*(x) = 1$; we cannot use it because uniform distributions have infinite variance.

In [31]:
ws.render_section('q14_24_zero_variance_and_sign_problem')

**Q24 · medium** · 💡

Suppose $f(x)$ is an oscillatory function on $[-1, 1]$ such that $\int_{-1}^0 f(x)\,dx = -5$ (where $f(x) \le 0$) and $\int_0^1 f(x)\,dx = +7$ (where $f(x) \ge 0$), so the true integral is $I = +2$ and $\int_{-1}^1 |f(x)|\,dx = 12$. If you were somehow able to sample from the **exact optimal distribution** $p^*(x) = |f(x)| / 12$, what would be the variance $\sigma^2 = \langle (f(x)/p^*(x) - I)^2 \rangle_{p^*}$ of a single Monte Carlo draw?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the discrete values taken by the ratio $f(x)/p^*(x)$ for positive and negative regions of $f(x)$, and calculate $\langle g^2 \rangle - I^2$.

</div>
</details>

- [ ] **(A)** $\sigma^2 = 12^2 - 2^2 = 140$ (non-zero! Because $f(x)$ changes sign, the estimator jumps between $+12$ and $-12$, illustrating the 'Sign Problem' where zero variance is impossible).
- [ ] **(B)** $\sigma^2 = 0$ (because $p^*(x)$ always achieves zero variance even when $f(x)$ changes sign).
- [ ] **(C)** $\sigma^2 = \infty$.
- [ ] **(D)** $\sigma^2 = 2$.

In [32]:
ws.render_section('q14_25_gaussian_is_finite_variance_bound')

**Q25 · medium** · 💡

In the lecture's Gaussian importance sampling example, we integrate $f(x) = \frac{e^{-x^2/2}}{1+x^2}$ using the trial Gaussian density $p_a(x) = \frac{e^{-x^2/(2a)}}{\sqrt{2\pi a}}$. The estimator variance is:
$$\sigma^2(a) = \int_{-\infty}^{\infty} dx\,\frac{\sqrt{2\pi a}}{(1+x^2)^2}\exp\!\left[-x^2\left(1 - \frac{1}{2a}\right)\right] - I^2$$
What is the critical lower bound $a_{\text{crit}}$ such that $\sigma^2(a)$ is **finite** if and only if $a > a_{\text{crit}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the condition on the Gaussian exponent that prevents the integrand from diverging at asymptotic distances $\pm\infty$.

</div>
</details>

_Numeric answer required._

In [33]:
ws.render_section('q14_26_gaussian_is_optimal_a')

**Q26 · easy** · 💡

In the same lecture example $f(x) = \frac{e^{-x^2/2}}{1+x^2}$ with $p_a(x) = \frac{e^{-x^2/(2a)}}{\sqrt{2\pi a}}$, the "obvious" choice matching the Gaussian numerator is $a = 1.0$. However, because the Lorentzian denominator $\frac{1}{1+x^2}$ narrows the peak of $f(x)$ near $x = 0$, what **optimal value** of $a$ (quoted ) minimizes the variance and relative CPU time?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Optimal value a = 0.6' or inspect the green minimum marker on the Variance Curve view in the interactive lab above.

</div>
</details>

_Numeric answer required._

In [34]:
ws.render_section('q14_27_infinite_variance_visual_signature')

**Q27 · easy** · 💡

of the reference notes (*"What does infinite variance look like?"*), how does a Monte Carlo time-series trace of the estimator $g(x_i) = f(x_i)/p_a(x_i)$ behave visually when $a = 0.2$ or $a = 0.4$ (in the infinite-variance regime $a \le 0.5$), compared to the near-optimal regime $a = 0.6$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Set $a = 0.25$ vs. $a = 0.60$ in the interactive Importance Sampling lab above and compare the traces.

</div>
</details>

- [ ] **(A)** For $a \le 0.5$, the trace exhibits violent upward 'spikes' and heavily skewed long-tailed histograms whenever a rare draw lands in the tail where $p_a(x) \ll f(x)$; for $a = 0.6$, the trace fluctuates symmetrically within a narrow, bounded band.
- [ ] **(B)** For $a \le 0.5$, the estimator trace is a perfectly flat horizontal line with zero noise.
- [ ] **(C)** For $a \le 0.5$, every single sample $f(x_i)/p_a(x_i)$ is negative.
- [ ] **(D)** The traces for $a = 0.2$ and $a = 0.6$ look visually identical.

In [35]:
ws.render_section('q14_28_undersampling_vs_oversampling')

**Q28 · medium** · 💡

Why is **over-sampling** the tails of an integrand (choosing an importance distribution $p(x)$ that is slightly broader/heavier-tailed than $|f(x)|$, such as $a = 1.0$) always much safer than **under-sampling** the tails (choosing $p(x)$ narrower than $\sqrt{|f(x)|}$, such as $a = 0.4$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Do not under-sample -- could cause infinite variance. Over-sampling -- loss in efficiency but not infinite variance.'.

</div>
</details>

- [ ] **(A)** Because under-sampling causes $f(x)/p(x)$ to blow up in the tails, risking infinite variance ($\sigma^2 = \infty$), whereas over-sampling keeps $f(x)/p(x) \to 0$ bounded in the tails and only incurs a moderate, finite loss in efficiency.
- [ ] **(B)** Because over-sampling violates conservation of energy.
- [ ] **(C)** Because under-sampling always reduces the variance to zero.
- [ ] **(D)** Because over-sampling makes the integral mean $\langle f/p \rangle_p$ depend on $a$.

In [36]:
ws.render_section('q14_29_power_law_importance_sampling_calc')

**Q29 · hard** · 💡

Let's calculate an exact variance reduction factor by hand! Suppose we want to integrate $f(x) = 3x^2$ on $[0, 1]$, whose exact integral is $I = \int_0^1 3x^2\,dx = 1$.
- With **uniform sampling** $p_{\text{unif}}(x) = 1$, the per-sample variance is $\sigma_{\text{unif}}^2 = \int_0^1 (3x^2)^2\,dx - 1^2 = \frac{9}{5} - 1 = 0.80$.
- Suppose instead we use **linear importance sampling** $p(x) = 2x$ on $[0, 1]$ (sampled via $x = \sqrt{u}$).
What is the exact per-sample variance $\sigma_{\text{IS}}^2 = \int_0^1 \frac{f(x)^2}{p(x)}\,dx - I^2$ using $p(x) = 2x$? (Enter as a decimal to 3 decimal places.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the second moment integral $\int_0^1 [f(x)^2 / p(x)] dx$ and subtract the squared true mean $I^2$.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('sec_correlated_and_boltzmann_sampling')

### Correlated Sampling for Differences & Functions of Integrals
Often we care less about an absolute integral $F_1 = \int dx\,f_1(x)$ than about a **small difference** (such as the energy difference $\Delta E = F_1 - F_2$ between two slightly perturbed crystal structures) or a general smooth function $G(F_1, F_2)$.
- If we evaluate $F_1$ and $F_2$ using **independent** random walks, their variances *add*: $\text{Var}(F_1 - F_2) = \text{Var}(F_1) + \text{Var}(F_2)$, drowning a tiny physical difference $\Delta E$ in statistical noise.
- **Correlated Sampling:** Evaluate *both* integrals using the **exact same importance distribution $p(x)$ and the exact same sequence of random points $\{x_i\}$**:
 $$\widehat{\Delta F} = \frac{1}{N}\sum_{i=1}^N \frac{f_1(x_i) - f_2(x_i)}{p(x_i)}$$
 Because the correlated noise cancels point-by-point when $f_1(x) \approx f_2(x)$, the variance of the difference shrinks dramatically! For a general function $G(F_1, F_2)$, linearizing $\delta G = \frac{\partial G}{\partial F_1}\delta F_1 + \frac{\partial G}{\partial F_2}\delta F_2$ shows that the **optimal importance function** is a weighted combination of the two integrands:
 $$p^*(x) \propto \left| f_1(x)\frac{\partial G}{\partial F_1} + f_2(x)\frac{\partial G}{\partial F_2} \right|$$

### Sampling the Canonical Boltzmann Distribution in $3N$ Dimensions
In statistical mechanics, we want to compute thermal averages of many observables $\{A_1(\mathbf{R}), A_2(\mathbf{R}), \dots\}$ across a $3N$-dimensional configuration space $\mathbf{R} = (\mathbf{r}_1, \dots, \mathbf{r}_N)$:
$$\langle A_i \rangle = \frac{\int d\mathbf{R}\,A_i(\mathbf{R})e^{-\beta V(\mathbf{R})}}{\int d\mathbf{R}\,e^{-\beta V(\mathbf{R})}} = \frac{\int d\mathbf{R}\,A_i(\mathbf{R})e^{-\beta V(\mathbf{R})}}{Z(\beta)}$$
Applying our optimal importance-sampling formula to $\langle A_i \rangle$ yields:
$$p_i^*(\mathbf{R}) \propto \left| e^{-\beta V(\mathbf{R})}\big(A_i(\mathbf{R}) - \langle A_i \rangle\big) \right|$$
Notice the two factors:
1. $e^{-\beta V(\mathbf{R})}$ varies by **hundreds of orders of magnitude** across configuration space and is **independent of the observable $A_i$**.
2. $|A_i(\mathbf{R}) - \langle A_i \rangle|$ varies slowly and depends on which observable we measure.

Because we want to calculate *many* physical properties from a single simulation trajectory, we choose the **Boltzmann distribution itself** as our universal importance sampling density:
$$p_{\text{Boltzmann}}(\mathbf{R}) = \frac{e^{-\beta V(\mathbf{R})}}{Z(\beta)}$$

### Why Independent ("Hit-or-Miss") Sampling of $e^{-\beta V(\mathbf{R})}$ Fails Exponentially at Large $N$
Why can't we just throw $N$ atoms at random uniform positions $\mathbf{R} \in [0, L]^{3N}$ ($P_{\text{unif}}(\mathbf{R}) = \Omega^{-N} = 1$) and average their Boltzmann weights $e^{-\beta V(\mathbf{R})}$ to estimate the partition function $Z(\beta) = \int d\mathbf{R}\,e^{-\beta V(\mathbf{R})} = e^{-\beta F(\beta)}$?

Let's compute the exact variance of the uniform-sampling partition function estimator:
$$\text{var}(Z) = \int d\mathbf{R}\,\left(e^{-\beta V(\mathbf{R})} - Z(\beta)\right)^2 = \int d\mathbf{R}\,e^{-2\beta V(\mathbf{R})} - Z(\beta)^2 = Z(2\beta) - Z(\beta)^2$$
Since $Z(\beta) = e^{-\beta F(\beta)}$ where $F(\beta)$ is the Helmholtz free energy, the relative variance (the variance of the dimensionless free energy $\beta F = -\ln Z$) is:
$$\text{var}(\beta F) = \frac{\text{var}(Z)}{Z(\beta)^2} = \frac{Z(2\beta)}{Z(\beta)^2} - 1 = \frac{e^{-2\beta F(2\beta)}}{\left(e^{-\beta F(\beta)}\right)^2} - 1 = e^{2\beta\left[F(\beta) - F(2\beta)\right]} - 1$$
Taking the square root gives the per-sample statistical error of $\beta F$:
$$\text{error}(\beta F) \approx \exp\!\Big[\beta\big(F(\beta) - F(2\beta)\big)\Big]$$

> [!WARNING]
> **Exponential Blowup in Particle Count $N$:** Because the Helmholtz free energy $F(\beta) = N f(\beta)$ is an **extensive** thermodynamic quantity (proportional to the number of particles $N$, with $F(\beta) > F(2\beta)$ because entropy $S = -\partial F/\partial T > 0$), the exponent $\beta[F(\beta) - F(2\beta)] = c\,N$ grows **linearly with $N$**! Therefore, the relative variance and required number of independent samples blow up **exponentially in $N$** ($\sim e^{+c N}$), just as bad as a deterministic grid!

#### Intuitive Hard-Sphere Picture
- Throw $N$ atoms at random into a box of area $A$; suppose the probability that no two atoms overlap is $q < 1$ (say $q = 10^{-3}$).
- Throw $2N$ atoms into a box of area $2A$ (same density!): both halves must simultaneously avoid overlap, so the success probability is $q^2$.
- Throw $mN$ atoms into a box of area $mA$: the probability of generating even a single non-overlapping configuration is:
 $$p_{\text{success}} = q^m = \exp\!\big(m \ln q\big) = \exp(-\text{const}\cdot N)$$
To escape this exponential catastrophe, we must never draw independent random configurations from scratch—instead, we use **Markov chain random walks** (Metropolis Monte Carlo) to stay inside the tiny, exponentially peaked region where $e^{-\beta V(\mathbf{R})}$ is large!


In [38]:
ws.render_section('q14_30_correlated_sampling_optimal_p')

**Q30 · medium** · 💡

Suppose you want to compute the difference $G(F_1, F_2) = F_1 - F_2$ between two integrals $F_1 = \int dx\,f_1(x)$ and $F_2 = \int dx\,f_2(x)$ using **correlated sampling** (evaluating both integrals with the same random points $\{x_i\}$ drawn from a single distribution $p(x)$). According to the lecture formula $p^*(x) \propto \left|f_1(x)\frac{\partial G}{\partial F_1} + f_2(x)\frac{\partial G}{\partial F_2}\right|$, what is the optimal importance sampling distribution $p^*(x)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

For $G(F_1, F_2) = F_1 - F_2$, what are $\partial G / \partial F_1$ and $\partial G / \partial F_2$?

</div>
</details>

- [ ] **(A)** $p^*(x) \propto |f_1(x) - f_2(x)|$ (sampling most heavily in regions where the difference between the two integrands is largest).
- [ ] **(B)** $p^*(x) \propto f_1(x)^2 + f_2(x)^2$.
- [ ] **(C)** $p^*(x) = 1$ (uniform sampling is always optimal for differences).
- [ ] **(D)** $p^*(x) \propto |f_1(x) / f_2(x)|$.

In [39]:
ws.render_section('q14_31_correlated_sampling_ratio_optimal_p')

**Q31 · hard** · 💡

In statistical mechanics, an ensemble average is a **ratio** of two integrals: $G(F_1, F_2) = \frac{F_1}{F_2}$, where $F_1 = \int dr\,A(r)e^{-\beta V(r)}$ and $F_2 = \int dr\,e^{-\beta V(r)}$ (so $\langle A \rangle = F_1 / F_2$). Applying the correlated sampling formula $p^*(r) \propto \left|f_1(r)\frac{\partial G}{\partial F_1} + f_2(r)\frac{\partial G}{\partial F_2}\right|$ with $\frac{\partial G}{\partial F_1} = \frac{1}{F_2}$ and $\frac{\partial G}{\partial F_2} = -\frac{F_1}{F_2^2} = -\frac{\langle A \rangle}{F_2}$ yields which optimal distribution $p^*(r)$ shown?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $f_1(r) = A(r)e^{-\beta V(r)}$ and $f_2(r) = e^{-\beta V(r)}$ into $|f_1/F_2 - f_2 \langle A \rangle / F_2|$ and factor out $e^{-\beta V(r)} / F_2$.

</div>
</details>

- [ ] **(A)** $p^*(r) \propto \left| e^{-\beta V(r)}\big(A(r) - \langle A \rangle\big) \right|$
- [ ] **(B)** $p^*(r) \propto e^{+\beta V(r)} / A(r)$
- [ ] **(C)** $p^*(r) \propto \left| A(r) + \langle A \rangle \right|$ (independent of $V(r)$)
- [ ] **(D)** $p^*(r) \propto e^{-2\beta V(r)}$

In [40]:
ws.render_section('q14_32_why_sample_boltzmann_factor_only')

**Q32 · easy** · 💡

Even though the optimal importance density for a single observable $A_i(\mathbf{R})$ is $p_i^*(\mathbf{R}) \propto |e^{-\beta V(\mathbf{R})}(A_i(\mathbf{R}) - \langle A_i \rangle)|$, why do standard canonical Monte Carlo simulations sample $p(\mathbf{R}) \propto e^{-\beta V(\mathbf{R})}$ (the first factor alone) rather than including $|A_i(\mathbf{R}) - \langle A_i \rangle|$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the bullet points of the reference notes.

</div>
</details>

- [ ] **(A)** Because we want to measure a whole set of different physical averages $\langle A_1 \rangle, \langle A_2 \rangle, \dots$ from the same trajectory, and $e^{-\beta V(\mathbf{R})}$ is the universal, overwhelmingly peaked factor independent of $A_i$ (and avoids zero-density nodes where $A_i = \langle A_i \rangle$).
- [ ] **(B)** Because $A_i(\mathbf{R}) - \langle A_i \rangle$ is always equal to $1$ for every configuration $\mathbf{R}$.
- [ ] **(C)** Because $e^{-\beta V(\mathbf{R})}$ is a flat uniform distribution.
- [ ] **(D)** Because $\langle A_i \rangle$ is always known exactly before the simulation starts.

In [41]:
ws.render_section('q14_33_partition_function_variance_formula')

**Q33 · medium** · 💡

If you attempt to estimate the canonical partition function $Z(\beta) = \int d\mathbf{R}\,e^{-\beta V(\mathbf{R})} = e^{-\beta F(\beta)}$ by sampling configurations $\mathbf{R}$ **independently and uniformly** in the box ($P(\mathbf{R}) = 1$), what is the exact expression for the relative variance $\text{var}(\beta F) = \frac{\text{var}(Z)}{Z(\beta)^2}$ derived?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $\int d\mathbf{R}\,(e^{-\beta V(\mathbf{R})})^2 = \int d\mathbf{R}\,e^{-2\beta V(\mathbf{R})} = Z(2\beta) = e^{-2\beta F(2\beta)}$.

</div>
</details>

- [ ] **(A)** $\text{var}(\beta F) = \frac{Z(2\beta)}{Z(\beta)^2} - 1 = e^{2\beta[F(\beta) - F(2\beta)]} - 1$
- [ ] **(B)** $\text{var}(\beta F) = \frac{1}{N}$
- [ ] **(C)** $\text{var}(\beta F) = 2\beta F(\beta)$
- [ ] **(D)** $\text{var}(\beta F) = 0$

In [42]:
ws.render_section('q14_34_free_energy_extensivity_blowup')

**Q34 · medium** · 💡

Why does the relative error $\text{error}(\beta F) = \exp\!\big[\beta(F(\beta) - F(2\beta))\big]$ of uniform independent Boltzmann sampling blow up **exponentially** with the number of particles $N$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What thermodynamic scaling property does the Helmholtz free energy $F(\beta)$ satisfy as system size $N$ increases?

</div>
</details>

- [ ] **(A)** Because the Helmholtz free energy $F(\beta) = N f(\beta)$ is an extensive quantity (proportional to $N$, with $F(\beta) > F(2\beta)$), so the exponent $\beta[F(\beta) - F(2\beta)] \propto N$ grows linearly with $N$.
- [ ] **(B)** Because the inverse temperature $\beta = 1/(k_B T)$ is proportional to $N^3$.
- [ ] **(C)** Because $F(\beta)$ is always equal to $F(2\beta)$.
- [ ] **(D)** Because uniform random number generators stop working when $N > 12$.

In [43]:
ws.render_section('q14_35_hard_sphere_overlap_scaling')

**Q35 · medium** · 💡

In the lecture's intuitive hard-sphere explanation, suppose placing $N = 10$ atoms at random into a sub-box of area $A$ has a non-overlap probability of $q = 0.10$. If you scale the system at constant density to $m = 10$ such independent sub-boxes ($100$ atoms total), what is $\log_{10}(1 / p)$, the base-10 logarithm of the average number of random configurations required to obtain a single non-overlapping state?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply the number of sub-boxes by the logarithm of the inverse success probability per sub-box.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q14_36_partition_function_error_numeric')

**Q36 · hard** · 💡

Consider a system of $N = 50$ particles where the extensive free-energy difference per particle between $\beta$ and $2\beta$ is $\beta[f(\beta) - f(2\beta)] = 0.20$. Using the formula $\text{error}(\beta F) = \exp\!\big[\beta(F(\beta) - F(2\beta))\big]$ with $F = N f$, what is the natural logarithm $\ln[\text{error}(\beta F)]$ of the single-sample statistical error of $\beta F$ under independent uniform sampling?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the product of the total particle number and the per-particle free-energy difference parameter.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q14_37_debugging_importance_sampling')

**Q37 · easy** · 💡

According to of the reference notes, what is the essential **debugging check** you should always perform after implementing a new importance sampling distribution $p_a(x)$ in a Monte Carlo integration code?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that the mean of $\hat{I} = \langle f(x)/p_a(x) \rangle_{p_a}$ should not depend on $a$!

</div>
</details>

- [ ] **(A)** Verify that the estimated integral value $\hat{I}$ remains statistically independent of the importance sampling distribution parameters (e.g., varying $a$ across several valid values yields the same mean within error bars).
- [ ] **(B)** Verify that the variance $\sigma^2(a)$ is completely independent of $a$.
- [ ] **(C)** Verify that setting $a = -1$ produces positive probabilities.
- [ ] **(D)** Verify that $\hat{I}$ increases linearly with the parameter $a$.

In [46]:
ws.render_section('q14_38_box_muller_single_pair_calc')

**Q38 · medium** · 💡

Using the standard trigonometric Box–Muller transform:
$$y_1 = \sqrt{-2\ln x_1}\cos(2\pi x_2), \qquad y_2 = \sqrt{-2\ln x_1}\sin(2\pi x_2)$$
if the uniform PRNG draws $x_1 = e^{-2} \approx 0.135335$ and $x_2 = 0.25$, what is the exact value of the **second** generated normal variate $y_2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the radial envelope term at $x_1 = e^{-2}$ and multiply by the sine of the angle corresponding to $x_2 = 0.25$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q14_39_rejection_undersampling_bug')

**Q39 · medium** · 💡

In von Neumann rejection sampling with acceptance ratio $c(x) = \frac{p(x)}{a\,q(x)}$, what happens if you choose the envelope constant $a$ **too small** such that $a\,q(x) < p(x)$ (i.e., $c(x) > 1$) near the peak of $p(x)$, or if you choose $q(x)$ way too small in a region where $p(x)$ is finite?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can a probability $u \le c(x)$ with $u \in [0, 1]$ ever exceed $1.0$ if $c(x) > 1$?

</div>
</details>

- [ ] **(A)** If $a\,q(x) < p(x)$ ($c(x) > 1$), the acceptance probability is clipped at $1.0$, systematically truncating the peak of $p(x)$ and biasing the sampled distribution; conversely, if $q(x)$ under-samples a region ($q(x) \ll p(x)$), $a$ must be made enormous, collapsing the overall acceptance efficiency $1/a$.
- [ ] **(B)** Setting $a < \max[p/q]$ increases the efficiency above $100\%$ while still sampling $p(x)$ without bias.
- [ ] **(C)** The rejection algorithm automatically rescales $u$ to $[0, 10]$ so no bias occurs.
- [ ] **(D)** The variance of the accepted points becomes zero.

In [48]:
ws.render_section('q14_40_cholesky_triangular_complexity')

**Q40 · medium** · 💡

In a $D = 10$-dimensional multivariate Gaussian sampler $\mathbf{x} = \mathbf{S}\mathbf{y}$, the Cholesky factor $\mathbf{S}$ is a $10 \times 10$ lower-triangular matrix ($S_{\alpha i} = 0$ for all entries strictly above the main diagonal, $i > \alpha$). How many entries of $\mathbf{S}$ lie strictly above the diagonal and are guaranteed to be zero?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Count the number of off-diagonal elements in the upper triangular portion of a $D \times D$ square matrix.

</div>
</details>

_Numeric answer required._

In [49]:
ws.render_section('q14_41_poisson_exponential_product_rule')

**Q41 · hard** · 💡

Because the waiting times $y_k = -\ln(u_k)/a$ between consecutive events in a Poisson process of rate $a$ are independent exponential random variables, the number of events $n$ occurring in time interval $t$ (mean $\nu = at$) is the largest integer $n$ such that $\sum_{k=1}^n y_k \le t < \sum_{k=1}^{n+1} y_k$. Substituting $y_k = -\ln(u_k)/a$ and exponentiating both sides gives which fast multiplication-only rule for generating a Poisson integer $n \sim P_\nu(n)$ from uniform draws $u_k \sim U(0, 1)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Rewrite $\sum_{k=1}^{n+1} (-\ln u_k / a) > t$ as $\ln(\prod_{k=1}^{n+1} u_k) < -at$ and exponentiate!

</div>
</details>

- [ ] **(A)** Multiply independent uniform draws $u_1 u_2 \cdots$ until the product first drops below $e^{-\nu} = e^{-at}$ at step $n + 1$ ($\prod_{k=1}^n u_k \ge e^{-\nu} > \prod_{k=1}^{n+1} u_k$), and return $n$.
- [ ] **(B)** Add $12$ uniform draws and subtract $e^{-\nu}$.
- [ ] **(C)** Take $\max(u_1, \dots, u_n)$ and divide by $n!$.
- [ ] **(D)** Reject $u_k$ whenever $u_k > \nu$.

In [50]:
ws.render_section('q14_42_sampling_methods_synthesis')

**Q42 · medium** · 💡

Match each of the four canonical simulation sampling tasks below to the most appropriate and efficient algorithm covered in Lesson 14:
1. Selecting one of $K = 500$ discrete KMC hopping rates $\{p_k\}$.
2. Generating a $30$-dimensional correlated harmonic normal-mode displacement vector with covariance matrix $\mathbf{T}$.
3. Evaluating a 5D integral $\int d^5x\,f(\mathbf{x})$ where $f(\mathbf{x}) > 0$ has a known approximate analytic shape $p(\mathbf{x})$.
4. Sampling configurations of $N = 1000$ interacting Lennard-Jones atoms weighted by $e^{-\beta V(\mathbf{R})}$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Combine the key takeaways from Sections 1, 2, 3, and 4.

</div>
</details>

- [ ] **(A)** (1) Precomputed cumulative table $c_k$ with $\mathcal{O}(\log_2 K)$ bisection search; (2) Box–Muller standard normals transformed via Cholesky factor $\mathbf{x} = \mathbf{S}\mathbf{y}$ ($\mathbf{S}\mathbf{S}^T = \mathbf{T}$); (3) Importance sampling $\langle f(\mathbf{x})/p(\mathbf{x}) \rangle_p$ with non-undersampled tails; (4) Markov chain random walk (Metropolis MC) near the peak of $e^{-\beta V(\mathbf{R})}$.
- [ ] **(B)** (1) 500-dimensional rejection sampling; (2) 1D inverse CDF table lookup; (3) Uniform hit-or-miss sampling; (4) Independent uniform placement of all $1000$ atoms.
- [ ] **(C)** (1) Cholesky decomposition; (2) Bisection table search; (3) Geiger counter TRNG; (4) 12-udrn composition.
- [ ] **(D)** (1) Linear scan; (2) Rejection sampling in 30D; (3) Importance sampling with $a = 0.2$ (under-sampled tails); (4) Uniform grid quadrature.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()